# Data persistence, sampling, and audio extraction

Historical research source, original zero-based cells **11?16**. Outputs have been cleared.

These cells retain the original global state, Colab paths, manual annotations, and checkpoint assumptions. This notebook is not standalone; earlier state and external artifacts may be required. Some repair branches are alternatives. Read [the migration guide](../../docs/migration.md) before selecting cells to run. For a new baseline, use [the package workflow](../amharic_asr.ipynb).


## Original source cell 11

Position 12 in the original notebook; original code comments may use different numbering.


In [ ]:

from google.colab import drive
from pathlib import Path
import shutil
import json
import hashlib

# Mount Google Drive.
drive.mount("/content/drive")

# Create project directories.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Amharic_ASR_XLSR"
)

DATA_DIR = PROJECT_DIR / "data"
TOKENIZER_SAVE_DIR = PROJECT_DIR / "tokenizer"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

for directory in [
    DATA_DIR,
    TOKENIZER_SAVE_DIR,
    CHECKPOINT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# Save the original and normalized metadata.
files_to_save = {
    "train_original.jsonl": OUTPUT_PATH,
    "train_normalized.jsonl": NORMALIZED_PATH,
    "validation_normalized.jsonl": VAL_METADATA_PATH,
}

for name, source in files_to_save.items():
    destination = DATA_DIR / name
    shutil.copy2(source, destination)

    print(
        f"Saved {name}: "
        f"{destination.stat().st_size / 1024**2:.2f} MB"
    )

# Save the complete tokenizer directory.
shutil.copytree(
    TOKENIZER_DIR,
    TOKENIZER_SAVE_DIR,
    dirs_exist_ok=True
)

# Save experiment metadata.
experiment_config = {
    "dataset": "google/WaxalNLP",
    "dataset_config": "amh_asr",
    "model": "facebook/wav2vec2-xls-r-300m",
    "train_examples": 38022,
    "validation_examples": 2912,
    "vocab_size": len(tokenizer),
    "blank_token_id": tokenizer.pad_token_id,
    "target_sample_rate": 16000,
    "normalization": "NFC, lowercase Latin, strip punctuation and unsupported symbols, collapse whitespace",
}

with (PROJECT_DIR / "experiment_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        experiment_config,
        file,
        ensure_ascii=False,
        indent=2
    )

# Verify that the saved tokenizer can be reloaded.
from transformers import Wav2Vec2CTCTokenizer

saved_tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    str(TOKENIZER_SAVE_DIR)
)

assert saved_tokenizer.get_vocab() == tokenizer.get_vocab()

print("\nBACKUP COMPLETE")
print("Project:", PROJECT_DIR)
print("Tokenizer:", TOKENIZER_SAVE_DIR)
print("Vocabulary size:", len(saved_tokenizer))
print("Checkpoint directory:", CHECKPOINT_DIR)


## Original source cell 12

Position 13 in the original notebook; original code comments may use different numbering.


In [ ]:

from itertools import islice
from datasets import load_dataset
from torchcodec.decoders import AudioDecoder
from IPython.display import Audio, display
import numpy as np

def inspect_recordings(shard, count, split_name):
    stream = load_dataset(
        "parquet",
        data_files={"train": shard["url"]},
        split="train",
        streaming=True
    ).decode(False)

    for i, example in enumerate(islice(stream, count), 1):
        audio_info = example["audio"]

        source = audio_info.get("bytes")
        if source is None:
            source = audio_info.get("path")

        if source is None:
            raise ValueError("Audio bytes/path missing")

        # Decode AND resample to our model's required 16 kHz.
        decoder = AudioDecoder(
            source,
            sample_rate=16000,
            num_channels=1
        )

        samples = decoder.get_all_samples()
        waveform = samples.data[0].cpu().numpy()

        duration = len(waveform) / samples.sample_rate
        rms = float(np.sqrt(np.mean(waveform ** 2)))

        print(f"\n{'=' * 60}")
        print(f"{split_name} RECORDING {i}")
        print(f"{'=' * 60}")

        print("ID:", example["id"])
        print("Speaker:", example["speaker_id"])
        print("Duration:", round(duration, 2), "seconds")
        print("Decoded sample rate:", samples.sample_rate, "Hz")
        print("Audio RMS:", round(rms, 4))

        print("\nORIGINAL TRANSCRIPTION:")
        print(example["transcription"])

        print("\nNORMALIZED TRANSCRIPTION:")
        print(normalize_amharic_ctc(example["transcription"]))

        print("\nLISTEN:")
        display(Audio(waveform, rate=samples.sample_rate))


# Two training recordings.
inspect_recordings(
    train_shards[0],
    count=2,
    split_name="TRAIN"
)

# One validation recording.
inspect_recordings(
    val_shards[0],
    count=1,
    split_name="VALIDATION"
)


## Original source cell 13

Position 14 in the original notebook; original code comments may use different numbering.


In [ ]:

import random
import math
import statistics
import numpy as np
import fsspec
import pyarrow.parquet as pq

from torchcodec.decoders import AudioDecoder
from tqdm.auto import tqdm

SEED = 42
rng = random.Random(SEED)

TARGET_HOURS = 25

# Sample 9 shards distributed throughout the training dataset.
shard_indices = sorted(set(
    round(i * (len(train_shards) - 1) / 8)
    for i in range(9)
))

duration_probe = []
errors = []

for shard_index in tqdm(
    shard_indices,
    desc="Sampling audio across shards"
):
    shard = train_shards[shard_index]

    try:
        with fsspec.open(
            shard["url"],
            "rb",
            block_size=8 * 1024**2
        ) as file:
            parquet = pq.ParquetFile(file)

            # Select a random row group rather than always
            # reading the beginning of each shard.
            group_index = rng.randrange(
                parquet.metadata.num_row_groups
            )

            table = parquet.read_row_group(
                group_index,
                columns=["id", "speaker_id", "audio"]
            )

            records = table.to_pylist()

            chosen_indices = rng.sample(
                range(len(records)),
                min(5, len(records))
            )

            for index in chosen_indices:
                record = records[index]
                audio = record["audio"]

                source = audio.get("bytes")
                if source is None:
                    source = audio.get("path")

                decoder = AudioDecoder(source)
                duration = decoder.metadata.duration_seconds

                if duration is None or not math.isfinite(duration):
                    continue

                duration_probe.append({
                    "id": record["id"],
                    "speaker_id": record["speaker_id"],
                    "duration": float(duration),
                    "shard_index": shard_index
                })

    except Exception as exc:
        errors.append((shard_index, str(exc)[:200]))


durations = np.array([
    item["duration"] for item in duration_probe
])

assert len(durations) > 0, "No valid audio durations found."

print("\nAUDIO DURATION ANALYSIS")
print("-" * 48)

print("Recordings inspected:", len(durations))
print(
    "Unique speakers:",
    len(set(item["speaker_id"] for item in duration_probe))
)

print("Mean:", round(float(durations.mean()), 2), "seconds")
print("Median:", round(float(np.median(durations)), 2), "seconds")
print("Minimum:", round(float(durations.min()), 2), "seconds")
print("Maximum:", round(float(durations.max()), 2), "seconds")

print("\nDURATION PERCENTILES")
for percentile in [10, 25, 50, 75, 90, 95]:
    value = np.percentile(durations, percentile)
    print(f"P{percentile}: {value:.2f} seconds")

print("\nDURATION BUCKETS")
buckets = [
    ("Under 10s", durations < 10),
    ("10-20s", (durations >= 10) & (durations < 20)),
    ("20-30s", (durations >= 20) & (durations < 30)),
    ("30-40s", (durations >= 30) & (durations < 40)),
    ("40s+", durations >= 40)
]

for label, mask in buckets:
    print(f"{label:12} {int(mask.sum()):3} recordings")

estimated_total_hours = (
    38022 * durations.mean() / 3600
)

estimated_recordings = (
    TARGET_HOURS * 3600 / durations.mean()
)

print("\nPRELIMINARY TRAINING ESTIMATES")
print(
    "Estimated full training duration:",
    round(estimated_total_hours, 1),
    "hours"
)
print(
    "Estimated recordings for 25 hours:",
    round(estimated_recordings)
)

print("\nRead errors:", len(errors))
for error in errors[:5]:
    print(error)


## Original source cell 14

Position 15 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import math
import random
from pathlib import Path
from collections import defaultdict, Counter

# =============================================
# EXPERIMENT CONFIGURATION
# =============================================

TRAIN_TARGET_HOURS = 25.0   # Change to 50, 100, etc.
                            # Use None for the full corpus.

RANDOM_SEED = 42

# Estimated from our 43-recording audio probe.
ESTIMATED_CLIP_SECONDS = 18.26

# Extra candidates to allow filtering and duration variation.
CANDIDATE_BUFFER = 1.20

# =============================================
# BUILD STABLE, SPEAKER-BALANCED PRIORITY ORDER
# =============================================

rng = random.Random(RANDOM_SEED)
speaker_pools = defaultdict(list)

with NORMALIZED_PATH.open(encoding="utf-8") as file:
    for row_index, line in enumerate(file):
        record = json.loads(line)
        record["source_row_index"] = row_index
        speaker_pools[record["speaker_id"]].append(record)

# Reproducible shuffling within each speaker.
for speaker in sorted(speaker_pools):
    rng.shuffle(speaker_pools[speaker])

active_speakers = sorted(speaker_pools)
priority_records = []

# Round-robin selection gives speakers opportunities
# to contribute recordings before others dominate.
while active_speakers:
    rng.shuffle(active_speakers)
    next_active = []

    for speaker in active_speakers:
        pool = speaker_pools[speaker]

        if pool:
            record = pool.pop()
            record["selection_rank"] = len(priority_records)
            priority_records.append(record)

        if pool:
            next_active.append(speaker)

    active_speakers = next_active

assert len(priority_records) == 38022
assert len({r["id"] for r in priority_records}) == 38022

# =============================================
# SELECT A CANDIDATE POOL
# =============================================

if TRAIN_TARGET_HOURS is None:
    candidate_count = len(priority_records)
    plan_label = "full"
else:
    estimated_count = (
        TRAIN_TARGET_HOURS * 3600
        / ESTIMATED_CLIP_SECONDS
    )

    candidate_count = min(
        math.ceil(estimated_count * CANDIDATE_BUFFER),
        len(priority_records)
    )
    plan_label = f"{TRAIN_TARGET_HOURS:g}h".replace(".", "p")

candidates = priority_records[:candidate_count]

# =============================================
# SAVE MANIFESTS AND CONFIGURATION
# =============================================

PRIORITY_PATH = DATA_DIR / "train_priority_all.jsonl"
CANDIDATE_PATH = (
    DATA_DIR / f"train_candidates_{plan_label}.jsonl"
)

def save_jsonl(records, path):
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

save_jsonl(priority_records, PRIORITY_PATH)
save_jsonl(candidates, CANDIDATE_PATH)

training_plan = {
    "target_hours": TRAIN_TARGET_HOURS,
    "seed": RANDOM_SEED,
    "estimated_clip_seconds": ESTIMATED_CLIP_SECONDS,
    "candidate_buffer": CANDIDATE_BUFFER,
    "candidate_count": candidate_count,
    "priority_manifest": str(PRIORITY_PATH),
    "candidate_manifest": str(CANDIDATE_PATH),
    "selection_method": "seeded_speaker_round_robin",
    "duration_status": "estimated_not_measured"
}

PLAN_PATH = DATA_DIR / f"training_plan_{plan_label}.json"

with PLAN_PATH.open("w", encoding="utf-8") as file:
    json.dump(training_plan, file, indent=2)

# =============================================
# REPORT
# =============================================

speaker_counts = Counter(
    r["speaker_id"] for r in candidates
)

candidate_characters = set().union(
    *(set(r["transcription"]) for r in candidates)
)

print("CONFIGURABLE TRAINING PLAN")
print("-" * 50)

print("Target hours:", TRAIN_TARGET_HOURS)
print("Total available recordings:", len(priority_records))
print("Candidate recordings:", len(candidates))
print("Selected speakers:", len(speaker_counts))
print("Minimum clips per speaker:", min(speaker_counts.values()))
print("Maximum clips per speaker:", max(speaker_counts.values()))

print(
    "Estimated candidate-pool hours:",
    round(
        len(candidates) * ESTIMATED_CLIP_SECONDS / 3600,
        2
    )
)

print(
    "Candidate character coverage:",
    len(candidate_characters),
    "/",
    len(normalized_counts)
)

print("\nSaved priority manifest:", PRIORITY_PATH)
print("Saved candidate manifest:", CANDIDATE_PATH)
print("Saved training plan:", PLAN_PATH)

print("\nNEXT: Measure actual candidate durations.")


## Original source cell 15

Position 16 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import math
from bisect import bisect_left
from pathlib import Path

import fsspec
import pyarrow.parquet as pq
from torchcodec.decoders import AudioDecoder
from tqdm.auto import tqdm

# ==========================================
# CONFIGURATION
# ==========================================

AUDIO_DIR = Path("/content/amharic_asr_candidate_audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_PATH = Path(
    "/content/amharic_audio_extraction_progress.jsonl"
)

TARGET_SECONDS = TRAIN_TARGET_HOURS * 3600

# Load selected candidates.
with CANDIDATE_PATH.open(encoding="utf-8") as f:
    candidate_records = [json.loads(line) for line in f]

candidates_by_row = {
    r["source_row_index"]: r for r in candidate_records
}

selected_positions = sorted(candidates_by_row)

# Restore extraction progress, ignoring missing local files.
completed = {}

if PROGRESS_PATH.exists():
    with PROGRESS_PATH.open(encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            audio_path = AUDIO_DIR / record["audio_file"]

            if audio_path.exists() and audio_path.stat().st_size > 0:
                completed[record["id"]] = record

print("Candidates:", len(candidate_records))
print("Already extracted:", len(completed))

errors = []
global_start = 0

# ==========================================
# EXTRACT SELECTED AUDIO FROM PARQUET
# ==========================================

with PROGRESS_PATH.open("a", encoding="utf-8") as progress_file:

    for shard_index, shard in enumerate(
        tqdm(train_shards, desc="Extracting audio shards")
    ):
        with fsspec.open(
            shard["url"],
            "rb",
            block_size=16 * 1024**2
        ) as source:
            parquet = pq.ParquetFile(source)

            shard_rows = parquet.metadata.num_rows
            shard_end = global_start + shard_rows

            # Find candidates belonging to this shard.
            left = bisect_left(selected_positions, global_start)
            right = bisect_left(selected_positions, shard_end)

            shard_positions = selected_positions[left:right]
            shard_position_set = set(shard_positions)

            group_start = global_start

            for group_index in range(parquet.metadata.num_row_groups):
                group_rows = parquet.metadata.row_group(
                    group_index
                ).num_rows

                group_end = group_start + group_rows

                needed_positions = [
                    pos for pos in shard_positions
                    if group_start <= pos < group_end
                    and candidates_by_row[pos]["id"] not in completed
                ]

                if not needed_positions:
                    group_start = group_end
                    continue

                needed = set(needed_positions)
                row_offset = 0

                for batch in parquet.iter_batches(
                    row_groups=[group_index],
                    columns=["id", "audio"],
                    batch_size=64
                ):
                    records = batch.to_pylist()

                    for local_index, record in enumerate(records):
                        global_index = (
                            group_start + row_offset + local_index
                        )

                        if global_index not in needed:
                            continue

                        expected = candidates_by_row[global_index]

                        # Verify our transcript/audio indexing.
                        assert record["id"] == expected["id"], (
                            f"ID mismatch at row {global_index}"
                        )

                        try:
                            encoded_audio = record["audio"]["bytes"]

                            if encoded_audio is None:
                                raise ValueError(
                                    "Audio bytes missing from Parquet"
                                )

                            decoder = AudioDecoder(encoded_audio)
                            duration = (
                                decoder.metadata.duration_seconds
                            )

                            if (
                                duration is None
                                or not math.isfinite(duration)
                                or duration <= 0
                            ):
                                raise ValueError(
                                    f"Invalid duration: {duration}"
                                )

                            filename = expected["id"] + ".mp3"
                            destination = AUDIO_DIR / filename
                            temporary = AUDIO_DIR / (filename + ".tmp")

                            temporary.write_bytes(encoded_audio)
                            temporary.replace(destination)

                            result = {
                                **expected,
                                "duration_seconds": float(duration),
                                "audio_file": filename
                            }

                            completed[expected["id"]] = result

                            progress_file.write(
                                json.dumps(
                                    result, ensure_ascii=False
                                ) + "\n"
                            )
                            progress_file.flush()

                        except Exception as exc:
                            errors.append(
                                (expected["id"], str(exc)[:200])
                            )

                    row_offset += len(records)

                group_start = group_end

            global_start = shard_end

# ==========================================
# SELECT FINAL TRAINING HOURS
# ==========================================

measured = sorted(
    completed.values(),
    key=lambda r: r["selection_rank"]
)

final_records = []
actual_seconds = 0.0

for record in measured:
    if actual_seconds >= TARGET_SECONDS:
        break

    final_records.append(record)
    actual_seconds += record["duration_seconds"]

FINAL_MANIFEST = (
    DATA_DIR / f"train_final_{TRAIN_TARGET_HOURS:g}h.jsonl"
)

with FINAL_MANIFEST.open("w", encoding="utf-8") as f:
    for record in final_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# ==========================================
# REPORT
# ==========================================

print("\nAUDIO EXTRACTION SUMMARY")
print("-" * 50)
print("Candidates requested:", len(candidate_records))
print("Candidates extracted:", len(completed))
print("Extraction errors:", len(errors))

print("\nFINAL TRAINING SELECTION")
print("Target hours:", TRAIN_TARGET_HOURS)
print("Actual hours:", round(actual_seconds / 3600, 3))
print("Selected recordings:", len(final_records))
print(
    "Selected speakers:",
    len({r["speaker_id"] for r in final_records})
)

print(
    "Local audio storage:",
    round(
        sum(p.stat().st_size for p in AUDIO_DIR.glob("*.mp3"))
        / 1024**3,
        2
    ),
    "GB"
)

print("\nFinal manifest:", FINAL_MANIFEST)
print("Audio directory:", AUDIO_DIR)

if errors:
    print("\nFirst extraction errors:")
    for error in errors[:5]:
        print(error)

if actual_seconds < TARGET_SECONDS:
    print(
        "\nWARNING: Candidate pool did not reach the target."
        "\nWe can expand the pool using the saved priority list."
    )
else:
    print("\nTarget duration achieved.")

assert global_start == 38022, (
    f"Dataset indexing mismatch: {global_start}"
)


## Original source cell 16

Position 17 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import tarfile
import hashlib
import shutil
from pathlib import Path
from tqdm.auto import tqdm

ARCHIVE_BATCH_SIZE = 600

ARCHIVE_DIR = PROJECT_DIR / "audio_archives" / "25h"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ARCHIVE_DIR = Path("/content/asr_archive_temp")
LOCAL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_INDEX_PATH = DATA_DIR / "train_audio_index_25h.jsonl"
ARCHIVE_INDEX_PATH = DATA_DIR / "audio_archive_index_25h.json"

# Get all extracted candidates in reproducible order.
records = sorted(
    completed.values(),
    key=lambda r: r["selection_rank"]
)

assert len(records) == len(candidate_records)

# Ensure every referenced MP3 exists before archiving.
for record in records:
    path = AUDIO_DIR / record["audio_file"]
    assert path.exists() and path.stat().st_size > 0, (
        f"Missing audio: {path}"
    )

# Save actual durations for ALL candidates.
with AUDIO_INDEX_PATH.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()

archive_index = {
    "target_hours": TRAIN_TARGET_HOURS,
    "total_recordings": len(records),
    "archives": []
}

batches = [
    records[i:i + ARCHIVE_BATCH_SIZE]
    for i in range(0, len(records), ARCHIVE_BATCH_SIZE)
]

for batch_number, batch in enumerate(
    tqdm(batches, desc="Backing up audio"), 1
):
    filename = f"amharic_audio_{batch_number:03d}.tar"

    local_tar = LOCAL_ARCHIVE_DIR / filename
    drive_tar = ARCHIVE_DIR / filename
    partial_tar = ARCHIVE_DIR / (filename + ".partial")

    # Create archive on Colab's faster local disk.
    with tarfile.open(local_tar, "w") as archive:
        for record in batch:
            source = AUDIO_DIR / record["audio_file"]
            archive.add(
                source,
                arcname=record["audio_file"],
                recursive=False
            )

    local_hash = sha256_file(local_tar)

    # Copy to Drive; rename only after copying completes.
    shutil.copyfile(local_tar, partial_tar)
    partial_tar.replace(drive_tar)

    # Verify the stored archive is identical.
    drive_hash = sha256_file(drive_tar)
    assert local_hash == drive_hash, (
        f"Checksum mismatch: {filename}"
    )

    archive_index["archives"].append({
        "filename": filename,
        "recordings": len(batch),
        "size_bytes": drive_tar.stat().st_size,
        "sha256": drive_hash
    })

    # Remove temporary TAR, keeping original MP3s.
    local_tar.unlink()

# Save archive index after successful verification.
with ARCHIVE_INDEX_PATH.open("w", encoding="utf-8") as f:
    json.dump(archive_index, f, indent=2)

total_bytes = sum(
    item["size_bytes"]
    for item in archive_index["archives"]
)

print("\nAUDIO BACKUP SUMMARY")
print("-" * 45)
print("Candidate recordings backed up:", len(records))
print("Archives created:", len(archive_index["archives"]))
print("Total archive size:", round(total_bytes / 1024**3, 2), "GB")
print("SHA-256 verification: PASSED")
print("Audio index:", AUDIO_INDEX_PATH)
print("Archive index:", ARCHIVE_INDEX_PATH)
print("Archive directory:", ARCHIVE_DIR)

print("\nOriginal local MP3 files remain available.")
